# Notebook 2: Machine Learning Model Training

**Flood Susceptibility Mapping — Karabük, Turkey**
CME434, Karabük University

**Purpose:** Train RF, XGBoost, SVM (2 param sets each), compare, save best model
**Input:**  `data/processed/Inputs.txt`, `data/processed/Label.txt`
**Output:** `outputs/models/best_flood_model.pkl`, `outputs/models/scaler.pkl`,
figures in `outputs/figures/`

In [ ]:
# !pip install xgboost --quiet  # uncomment in Colab if needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)
import xgboost as xgb
import joblib
import os

os.makedirs('../outputs/models',  exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)
print("All libraries loaded")

## Step 1 — Load Data

In [ ]:
X = np.loadtxt('../data/processed/Inputs.txt')
y = np.loadtxt('../data/processed/Label.txt')
print(f"X: {X.shape}  |  y: {y.shape}")
vals, cnts = np.unique(y, return_counts=True)
print(f"Classes: {dict(zip(vals.astype(int), cnts))}")

## Step 2 — Train/Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Flooded in test set: {int(y_test.sum())} / {len(y_test)}")

## Step 3 — Classifier 1: Random Forest

In [ ]:
print("RF Run 1 — default...")
rf1 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf1.fit(X_train, y_train)

print("RF Run 2 — tuned...")
rf2 = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=4,
    max_features='sqrt', random_state=42, n_jobs=-1)
rf2.fit(X_train, y_train)

for name, m in [('RF Default', rf1), ('RF Tuned', rf2)]:
    yp = m.predict(X_test)
    print(f"\n--- {name} ---")
    print(classification_report(y_test, yp,
          target_names=['Non-Flooded', 'Flooded']))
    print(f"AUC: {roc_auc_score(y_test, m.predict_proba(X_test)[:,1]):.4f}")

## Step 4 — Classifier 2: XGBoost

In [ ]:
print("XGB Run 1 — default...")
xgb1 = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5,
    random_state=42, eval_metric='logloss')
xgb1.fit(X_train, y_train)

print("XGB Run 2 — tuned...")
xgb2 = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=7,
    colsample_bytree=0.8, subsample=0.8, min_child_weight=3,
    random_state=42, eval_metric='logloss')
xgb2.fit(X_train, y_train)

for name, m in [('XGB Default', xgb1), ('XGB Tuned', xgb2)]:
    yp = m.predict(X_test)
    print(f"\n--- {name} ---")
    print(classification_report(y_test, yp,
          target_names=['Non-Flooded', 'Flooded']))
    print(f"AUC: {roc_auc_score(y_test, m.predict_proba(X_test)[:,1]):.4f}")

## Step 5 — Classifier 3: SVM (scaled features)

In [ ]:
print("SVM Run 1 — default (may take 1-2 min)...")
svm1 = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm1.fit(X_train_s, y_train)

print("SVM Run 2 — tuned...")
svm2 = SVC(kernel='rbf', C=10.0, gamma=0.01,
           probability=True, random_state=42)
svm2.fit(X_train_s, y_train)

for name, m in [('SVM Default', svm1), ('SVM Tuned', svm2)]:
    yp = m.predict(X_test_s)
    print(f"\n--- {name} ---")
    print(classification_report(y_test, yp,
          target_names=['Non-Flooded', 'Flooded']))
    print(f"AUC: {roc_auc_score(y_test, m.predict_proba(X_test_s)[:,1]):.4f}")

## Step 6 — Full Model Comparison Table

In [ ]:
models = {
    'RF Default':  (rf1,  X_test,   X_test),
    'RF Tuned':    (rf2,  X_test,   X_test),
    'XGB Default': (xgb1, X_test,   X_test),
    'XGB Tuned':   (xgb2, X_test,   X_test),
    'SVM Default': (svm1, X_test_s, X_test_s),
    'SVM Tuned':   (svm2, X_test_s, X_test_s),
}

rows = []
for name, (m, _, Xte) in models.items():
    yp    = m.predict(Xte)
    yprob = m.predict_proba(Xte)[:, 1]
    rep   = classification_report(y_test, yp, output_dict=True)
    rows.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, yp), 4),
        'F1':        round(f1_score(y_test, yp), 4),
        'AUC':       round(roc_auc_score(y_test, yprob), 4),
        'Precision': round(float(rep['1']['precision']), 4),
        'Recall':    round(float(rep['1']['recall']),    4),
    })

results_df = pd.DataFrame(rows).sort_values('AUC', ascending=False)
print(results_df.to_string(index=False))
print(f"\nBest model: {results_df.iloc[0]['Model']}")

## Step 7 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, (name, (m, _, Xte)) in enumerate(models.items()):
    yp = m.predict(Xte)
    ConfusionMatrixDisplay.from_predictions(
        y_test, yp, ax=axes[i // 3][i % 3],
        display_labels=['Non-Flooded', 'Flooded'],
        colorbar=False)
    axes[i // 3][i % 3].set_title(name, fontsize=11)
plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrices.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/figures/confusion_matrices.png")

## Step 8 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
palette = ['#2196F3','#1565C0','#FF9800','#E65100','#4CAF50','#1B5E20']
for (name, (m, _, Xte)), col in zip(models.items(), palette):
    RocCurveDisplay.from_predictions(
        y_test, m.predict_proba(Xte)[:, 1],
        ax=ax, name=name, color=col)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_title('ROC Curves — Flood Susceptibility Models', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/figures/roc_curves.png")

## Step 9 — Feature Importance (Random Forest)

In [ ]:
rf_rows   = results_df[results_df['Model'].str.startswith('RF')]
best_rf   = rf2 if rf_rows.iloc[0]['Model'] == 'RF Tuned' else rf1

# Replace generic names with actual band names after confirming GEE export order:
# feat_names = ['Elevation','Slope','Aspect','Hillshade','FlowAcc',
#               'D_Rivers','TWI','DrainDensity','Rainfall','NDVI',
#               'LULC','SurfaceWater','Curvature','PopDensity','D_Roads']
feat_names = [f'Feature_{i+1}' for i in range(X.shape[1])]

imp = pd.Series(best_rf.feature_importances_, index=feat_names).sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(imp.index, imp.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Importance (Gini)', fontsize=11)
ax.set_title('Feature Importance — Random Forest', fontsize=13)
for b, v in zip(bars, imp.values):
    ax.text(v + 0.001, b.get_y() + b.get_height() / 2,
            f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/figures/feature_importance.png")

## Step 10 — Save Best Model

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name][0]

joblib.dump(best_model, '../outputs/models/best_flood_model.pkl')
joblib.dump(scaler,     '../outputs/models/scaler.pkl')

print(f"Best model : {best_name}")
print(f"AUC        : {results_df.iloc[0]['AUC']}")
print(f"F1         : {results_df.iloc[0]['F1']}")
print(f"Accuracy   : {results_df.iloc[0]['Accuracy']}")
print("\nSaved: outputs/models/best_flood_model.pkl")
print("Saved: outputs/models/scaler.pkl")

results_df.to_csv('../outputs/figures/model_comparison.csv', index=False)
print("Saved: outputs/figures/model_comparison.csv")
print("\nDone. Next: colab/03_susceptibility_mapping.ipynb")